# Notebook 08: Ensemble Analysis

**Learning objectives:**
- Parse correlator data files produced by `Propagator.py`
- Perform ensemble averaging over multiple configurations
- Apply jackknife resampling for error estimation
- Extract meson masses with statistical uncertainties

**Prerequisites:** Notebooks 07, correlator data from batch processing

**Data:** Run `scripts/run_propagators_batch.sh` on your generated configs first.
Set `data_dir` below to point to the directory containing your correlator files.

**Uses:** `jackknife.py` for error estimation

In [ ]:
from notebook_utils import setup_paths, plot_correlator, plot_effective_mass
REPO = setup_paths()

import os
import numpy as np
import matplotlib.pyplot as plt
import jackknife

## 1. Correlator Data Format

Each YAML file contains correlator measurements for one gauge configuration:
- PION_5 (pseudoscalar, $\Gamma = \gamma_5$)
- SIGMA (scalar, $\Gamma = \mathbb{1}$)
- RHO_X, RHO_Y, RHO_Z (vector, $\Gamma = \gamma_i$)

These files are produced by running `Propagator.py` with `--save-correlators --channel all`
on each gauge configuration in your ensemble.

In [ ]:
# The batch script produces results_spectrum_* directories, each with a correlators/ subfolder.
# We glob all .dat files across all result directories.
data_dir = os.path.join(REPO, "su2", "meson_correlator")

import glob
files = sorted(glob.glob(os.path.join(data_dir, "results_spectrum_*", "correlators", "pion_correlator_*.dat")))

print(f"Found {len(files)} pion correlator files")
if files:
    print(f"First: {files[0].split('results_spectrum_')[-1]}")
    print(f"Last:  {files[-1].split('results_spectrum_')[-1]}")

# Build a mapping: for each result dir, collect all channel .dat files
result_dirs = sorted(set(os.path.dirname(os.path.dirname(f)) for f in files))
print(f"\nFrom {len(result_dirs)} result directories")

<cell_type>markdown</cell_type>## 2. Parse the .dat Files

Each `results_spectrum_*/correlators/` directory has files like
`pion_correlator_*.dat`, `sigma_correlator_*.dat`, `rho_x_correlator_*.dat`, etc.

Format: comment lines starting with `#`, then `t  C(t)` data lines.

In [ ]:
def parse_dat_file(filepath):
    """Parse a .dat correlator file, return array of C(t) values."""
    values = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 2:
                try:
                    values.append(float(parts[1]))
                except ValueError:
                    pass
    return np.array(values)

# Channel name mapping: filename prefix -> channel label
CHANNEL_MAP = {
    'pion': 'PION_5',
    'sigma': 'SIGMA',
    'rho_x': 'RHO_X',
    'rho_y': 'RHO_Y',
    'rho_z': 'RHO_Z',
}

In [ ]:
# Parse one file as example
sample = parse_dat_file(files[0])
print(f"Pion correlator from first config: {len(sample)} time slices")
print()
for t, c in enumerate(sample):
    print(f"  t={t:2d}: C(t) = {c:+.6e}")

## 3. Load All Configurations

Parse all 40 correlator files to build an ensemble.

In [ ]:
# TRY: Set Lt to match your lattice (20 for both 6^3x20 and 8^3x20)
Lt = 20  # temporal extent
all_corrs = {}

for rdir in result_dirs:
    corr_dir = os.path.join(rdir, "correlators")
    for prefix, channel in CHANNEL_MAP.items():
        dat_files = glob.glob(os.path.join(corr_dir, f"{prefix}_correlator_*.dat"))
        for df in dat_files:
            data = parse_dat_file(df)
            if len(data) == Lt:
                if channel not in all_corrs:
                    all_corrs[channel] = []
                all_corrs[channel].append(data)

for ch in sorted(all_corrs.keys()):
    print(f"{ch:8s}: {len(all_corrs[ch])} configs, {len(all_corrs[ch][0])} time slices")

## 4. Ensemble Average

The physical correlator is the **expectation value** over gauge configurations:
$$\langle C(t) \rangle = \frac{1}{N_{\rm cfg}} \sum_{i=1}^{N_{\rm cfg}} C_i(t)$$

In [ ]:
# Ensemble average the pion correlator
pion_data = np.array(all_corrs['PION_5'])  # shape: (N_cfg, Lt)
N_cfg = pion_data.shape[0]

pion_avg = np.mean(pion_data, axis=0)
pion_std = np.std(pion_data, axis=0) / np.sqrt(N_cfg)

print(f"Ensemble size: {N_cfg} configs")
print(f"\nEnsemble-averaged pion correlator:")
for t in range(Lt):
    print(f"  t={t:2d}: C(t) = {pion_avg[t]:+.6e} +/- {pion_std[t]:.2e}")

In [ ]:
# Plot ensemble-averaged correlator
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

t = np.arange(Lt)
c_abs = np.abs(pion_avg)

# Log plot
ax1.errorbar(t, c_abs, yerr=np.abs(pion_std), fmt='o-', capsize=3,
             markersize=5, label=f'{N_cfg} configs')
ax1.set_yscale('log')
ax1.set_xlabel('t')
ax1.set_ylabel('|C(t)|')
ax1.set_title(r'Pion correlator (ensemble average)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Individual configs (faded) + average
for i in range(min(10, N_cfg)):
    ax2.semilogy(t, np.abs(pion_data[i]), 'b-', alpha=0.1)
ax2.semilogy(t, c_abs, 'ro-', markersize=5, label='Ensemble average')
ax2.set_xlabel('t')
ax2.set_ylabel('|C(t)|')
ax2.set_title('Individual configs (blue) vs average (red)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Effective Mass and Plateau

$$m_{\rm eff}(t) = \ln\frac{|C(t)|}{|C(t+1)|}$$

A constant plateau at intermediate $t$ gives the ground-state mass.

In [ ]:
c_abs = np.abs(pion_avg)
c_abs = np.maximum(c_abs, 1e-30)  # avoid log(0)
m_eff = np.log(c_abs[:-1] / c_abs[1:])
t_eff = np.arange(len(m_eff))

# Simple error propagation
m_eff_err = np.sqrt((pion_std[:-1] / c_abs[:-1])**2 +
                    (pion_std[1:] / c_abs[1:])**2)

plt.figure(figsize=(8, 5))
valid = np.isfinite(m_eff) & (m_eff > 0) & (m_eff < 5)
plt.errorbar(t_eff[valid], m_eff[valid], yerr=m_eff_err[valid],
             fmt='s-', capsize=4, markersize=6)

# TRY: Change t_min and t_max to see how sensitive the extracted mass is
#      to the fit window. A good plateau should be stable under small changes.
t_min, t_max = 3, 8
plat_mask = valid & (t_eff >= t_min) & (t_eff <= t_max)
if np.any(plat_mask):
    m_plat = np.mean(m_eff[plat_mask])
    plt.axhline(m_plat, color='r', ls='--',
                label=f'Plateau: $m_\\pi$ = {m_plat:.3f}')
    plt.axvspan(t_min, t_max, alpha=0.1, color='yellow')

plt.xlabel('t', fontsize=13)
plt.ylabel(r'$m_{\rm eff}(t)$', fontsize=13)
plt.title(r'Pion effective mass (ensemble average)')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Jackknife Error Analysis

The **jackknife** method provides a robust error estimate by
leave-one-out resampling: create $N$ subsets, each missing one config,
and compute the spread of results.

In [ ]:
def jackknife_effective_mass(corr_data, t):
    """Compute effective mass at time t with jackknife errors."""
    N = len(corr_data)
    jack_masses = np.zeros(N)

    for i in range(N):
        # Leave out config i
        jack_avg = np.mean(np.delete(corr_data, i, axis=0), axis=0)
        c_abs = np.abs(jack_avg)
        if c_abs[t] > 0 and c_abs[t+1] > 0:
            jack_masses[i] = np.log(c_abs[t] / c_abs[t+1])
        else:
            jack_masses[i] = np.nan

    valid = np.isfinite(jack_masses)
    mean = np.mean(jack_masses[valid])
    # Jackknife error: sqrt((N-1)/N * sum(f_i - f_mean)^2)
    err = np.sqrt((N-1) / N * np.sum((jack_masses[valid] - mean)**2))
    return mean, err

In [ ]:
# Jackknife effective mass for all time slices
jack_mass = []
jack_err = []

for t in range(Lt - 1):
    m, e = jackknife_effective_mass(pion_data, t)
    jack_mass.append(m)
    jack_err.append(e)

jack_mass = np.array(jack_mass)
jack_err = np.array(jack_err)

print("Jackknife effective masses:")
for t in range(Lt - 1):
    if np.isfinite(jack_mass[t]) and jack_mass[t] > 0:
        print(f"  t={t:2d}: m_eff = {jack_mass[t]:.4f} +/- {jack_err[t]:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
valid = np.isfinite(jack_mass) & (jack_mass > 0) & (jack_mass < 5)
t_arr = np.arange(Lt - 1)

plt.errorbar(t_arr[valid], jack_mass[valid], yerr=jack_err[valid],
             fmt='s-', capsize=4, markersize=6, label='Jackknife')

# Fit plateau
fit_mask = valid & (t_arr >= t_min) & (t_arr <= t_max)
if np.any(fit_mask):
    weights = 1.0 / jack_err[fit_mask]**2
    m_final = np.sum(jack_mass[fit_mask] * weights) / np.sum(weights)
    m_final_err = 1.0 / np.sqrt(np.sum(weights))
    plt.axhline(m_final, color='r', ls='--',
                label=f'$m_\\pi = {m_final:.4f} \\pm {m_final_err:.4f}$')
    plt.axvspan(t_min, t_max, alpha=0.1, color='yellow', label='Fit range')

plt.xlabel('t', fontsize=13)
plt.ylabel(r'$m_{\rm eff}(t)$', fontsize=13)
plt.title(r'Pion mass with jackknife errors')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if np.any(fit_mask):
    print(f"\nFinal pion mass: m_pi = {m_final:.4f} +/- {m_final_err:.4f}")

## 7. All Channels

Repeat the analysis for sigma and rho channels.

In [ ]:
results_all = {}

for channel in all_corrs.keys():
    ch_data = np.array(all_corrs[channel])
    if ch_data.shape[1] != Lt:
        continue

    # Jackknife at each t
    ch_mass = []
    ch_err = []
    for t in range(Lt - 1):
        m, e = jackknife_effective_mass(ch_data, t)
        ch_mass.append(m)
        ch_err.append(e)

    ch_mass = np.array(ch_mass)
    ch_err = np.array(ch_err)

    # Plateau fit
    valid = (np.isfinite(ch_mass) & (ch_mass > 0) & (ch_mass < 5)
             & (t_arr >= t_min) & (t_arr <= t_max))
    if np.any(valid):
        w = 1.0 / ch_err[valid]**2
        m_fit = np.sum(ch_mass[valid] * w) / np.sum(w)
        m_fit_err = 1.0 / np.sqrt(np.sum(w))
    else:
        m_fit, m_fit_err = np.nan, np.nan

    results_all[channel] = {'mass': m_fit, 'error': m_fit_err,
                            'm_eff': ch_mass, 'm_err': ch_err}
    print(f"{channel:8s}: M = {m_fit:.4f} +/- {m_fit_err:.4f}")

In [ ]:
# Spectrum plot
channels_plot = [ch for ch in results_all if np.isfinite(results_all[ch]['mass'])]
masses_plot = [results_all[ch]['mass'] for ch in channels_plot]
errors_plot = [results_all[ch]['error'] for ch in channels_plot]

plt.figure(figsize=(8, 5))
x_pos = np.arange(len(channels_plot))
plt.bar(x_pos, masses_plot, yerr=errors_plot, capsize=5, alpha=0.7,
        edgecolor='k', width=0.6)
plt.xticks(x_pos, channels_plot, fontsize=11)
plt.ylabel('Mass (lattice units)', fontsize=13)
plt.title('Meson spectrum from ensemble data', fontsize=13)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Exercises

1. Repeat the jackknife analysis for the SIGMA channel.
   Is $M_\sigma > M_\pi$ as expected?
2. Average the three rho components ($\rho_x, \rho_y, \rho_z$) and
   compare to the pion. Is $M_\rho > M_\pi$?
3. Try different plateau fit ranges (e.g. $t_{\rm min} = 4$, $t_{\rm max} = 7$).
   How sensitive is the extracted mass to the fit window?
4. Instead of the standard jackknife, try bootstrap resampling
   (resample $N$ configs with replacement, repeat 1000 times).
   Compare errors.
5. **Bootstrap implementation**: Implement bootstrap resampling explicitly:
   ```python
   def bootstrap_effective_mass(corr_data, t, n_boot=200):
       N = len(corr_data)
       boot_masses = []
       for _ in range(n_boot):
           idx = np.random.randint(0, N, size=N)
           boot_avg = np.mean(corr_data[idx], axis=0)
           c = np.abs(boot_avg)
           if c[t] > 0 and c[t+1] > 0:
               boot_masses.append(np.log(c[t] / c[t+1]))
       return np.mean(boot_masses), np.std(boot_masses)
   ```
   Compare bootstrap errors to jackknife at several time slices.
   They should agree within $\sim 10\%$.
6. **Error scaling**: If you have $N$ configs, the statistical error should
   scale as $1/\sqrt{N}$. Verify by computing the pion effective mass error
   using subsets of $N = 5, 10, 20, 40$ configs from your ensemble.
   Plot error vs $1/\sqrt{N}$ — is it linear?